In [5]:
import os
import numpy as np
import pandas as pd

# =============================================================================
# Catalog paths
# =============================================================================
catalog_paths = [
    "./Datasets/ComCat/ComCat_catalog.csv",
    "./Datasets/QTM/SaltonSea_catalog.csv",
    "./Datasets/QTM/SanJac_catalog.csv",
    "./Datasets/WHITE/WHITE_catalog.csv",
    "./Datasets/SCEDC/SCEDC_catalog.csv",
]

# =============================================================================
# Jitter parameters
# =============================================================================
np.random.seed(42)

# Spatial jitter (km): Uniform[-0.5, 0.5]
spatial_jitter_range = 1.0

# Temporal jitter (second): Uniform[-0.05, 0.05]
time_jitter_range = 0.1

# Convert seconds to days
time_jitter_range_days = time_jitter_range / (24 * 3600)

# =============================================================================
# Process each catalog
# =============================================================================
for catalog_path in catalog_paths:

    print("=" * 70)
    print(f"Processing: {catalog_path}")

    # -------------------------------------------------------------------------
    # Read catalog
    # -------------------------------------------------------------------------
    catalog = pd.read_csv(catalog_path)

    # Convert time column to datetime
    catalog["time"] = pd.to_datetime(catalog["time"])

    df_modified = catalog.copy()

    # -------------------------------------------------------------------------
    # Add time_days column (days since 1970-01-01)
    # -------------------------------------------------------------------------
    df_modified["time_days"] = (
        df_modified["time"] - pd.Timestamp("1970-01-01")
    ).dt.total_seconds() / (24 * 3600)

    # -------------------------------------------------------------------------
    # Jitter duplicated (x, y)
    # -------------------------------------------------------------------------
    iteration = 1

    while True:

        duplicates = df_modified[
            df_modified.duplicated(subset=["x", "y"], keep=False)
        ]

        if duplicates.empty:
            print("No duplicate (x, y) locations found.")
            break

        print(f"Iteration {iteration}: {len(duplicates)} duplicated locations found.")

        for idx in duplicates.index:

            df_modified.loc[idx, "x"] += np.random.uniform(
                -spatial_jitter_range / 2,
                 spatial_jitter_range / 2,
            )

            df_modified.loc[idx, "y"] += np.random.uniform(
                -spatial_jitter_range / 2,
                 spatial_jitter_range / 2,
            )

        iteration += 1

    # -------------------------------------------------------------------------
    # Jitter duplicated time_days
    # -------------------------------------------------------------------------
    iteration = 1

    while True:

        duplicates = df_modified[
            df_modified.duplicated(subset=["time_days"], keep=False)
        ]

        if duplicates.empty:
            print("No duplicate time_days found.")
            break

        print(f"Iteration {iteration}: {len(duplicates)} duplicated time_days found.")

        for idx in duplicates.index:

            df_modified.loc[idx, "time_days"] += np.random.uniform(
                -time_jitter_range_days / 2,
                 time_jitter_range_days / 2,
            )

        iteration += 1

    # -------------------------------------------------------------------------
    # Sort by time_days
    # -------------------------------------------------------------------------
    df_modified = (
        df_modified
        .sort_values("time_days")
        .reset_index(drop=True)
    )

    # -------------------------------------------------------------------------
    # Save as a new file
    # -------------------------------------------------------------------------
    save_path = os.path.splitext(catalog_path)[0] + "_jitter.csv"

    df_modified.to_csv(save_path, index=False)

    print(f"Saved to: {save_path}")

print("=" * 70)
print("All catalogs have been processed.")

Processing: ./Datasets/ComCat/ComCat_catalog.csv
Iteration 1: 2071 duplicated locations found.
No duplicate (x, y) locations found.
Iteration 1: 10 duplicated time_days found.
No duplicate time_days found.
Saved to: ./Datasets/ComCat/ComCat_catalog_jitter.csv
Processing: ./Datasets/QTM/SaltonSea_catalog.csv
Iteration 1: 4792 duplicated locations found.
No duplicate (x, y) locations found.
Iteration 1: 2 duplicated time_days found.
No duplicate time_days found.
Saved to: ./Datasets/QTM/SaltonSea_catalog_jitter.csv
Processing: ./Datasets/QTM/SanJac_catalog.csv
Iteration 1: 1935 duplicated locations found.
No duplicate (x, y) locations found.
No duplicate time_days found.
Saved to: ./Datasets/QTM/SanJac_catalog_jitter.csv
Processing: ./Datasets/WHITE/WHITE_catalog.csv
Iteration 1: 2594 duplicated locations found.
No duplicate (x, y) locations found.
No duplicate time_days found.
Saved to: ./Datasets/WHITE/WHITE_catalog_jitter.csv
Processing: ./Datasets/SCEDC/SCEDC_catalog.csv
Iteration 1: